# AgroPredict Colombia
## Optimización y evaluación de modelos predictivos con rezagos t-1 y t-2

**Despliegue de Soluciones Analíticas — Tercera entrega**

### Objetivo

Optimizar y comparar los modelos supervisados desarrollados en
`04.1_modelado_2periodos.ipynb` para estimar el rendimiento municipal del maíz en
Colombia (t/ha), usando como variables el rendimiento del semestre anterior (t-1),
el del mismo semestre del año anterior (t-2), el semestre, el departamento y el
sistema productivo, sin el año.

En la comparación previa, con hiperparámetros fijos, XGBoost obtuvo el menor RMSE
(0.519 t/ha en validación 2024), seguido de cerca por Random Forest (0.530 t/ha),
mientras que Ridge quedó por detrás (0.543 t/ha). Por eso este cuaderno optimiza
Random Forest y XGBoost.

### Metodología

1. Construir el modelo de referencia (persistencia) basado en el rendimiento del
   semestre anterior.
2. Implementar validación temporal con ventanas expansivas.
3. Optimizar hiperparámetros de Random Forest y XGBoost.
4. Comparar el desempeño mediante RMSE (métrica principal), MAE y R².
5. Evaluar el modelo seleccionado sobre 2025 y reentrenarlo con 2019–2025 para
   producción.
6. Registrar los experimentos en MLflow.

### Consideraciones metodológicas

- Se usan solo las filas con ambos rezagos disponibles. Como 2019 no tiene t-2, el
  entrenamiento efectivo empieza en 2020, y las ventanas de validación cruzada son:
  2020 → 2021, 2020–2021 → 2022 y 2020–2022 → 2023.
- Selección de hiperparámetros: promedio del RMSE en las tres ventanas.
- Validación: año 2024, para comparar los modelos optimizados.
- Prueba final: año 2025, reservado hasta seleccionar el modelo y evaluado una sola
  vez.
- Modelo de producción: el seleccionado, reentrenado con 2019–2025 con los mismos
  hiperparámetros y sin ajustes adicionales.
- Se usa el RMSE como métrica principal porque en cerca del 42 % de los registros el
  rendimiento se repite exactamente del semestre anterior, lo que favorece
  artificialmente el MAE de la persistencia.
- Se utilizan únicamente variables disponibles antes de la siembra y se conserva la
  separación temporal para evitar fuga de información.

In [1]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import mlflow

from dotenv import load_dotenv

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import ParameterGrid
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from xgboost import XGBRegressor

# Rutas del proyecto
PROYECTO = Path.home() / "agropredict-colombia"
DATOS = PROYECTO / "data" / "processed" / "features_rezago_1_2.parquet"

# Configuración de MLflow
load_dotenv(PROYECTO / "config" / ".env", override=True)

MLFLOW_URI = os.getenv("MLFLOW_TRACKING_URI")
MLFLOW_EXPERIMENTO = os.getenv("MLFLOW_EXPERIMENT")

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENTO)

print("Archivo de datos:", DATOS)
print("Datos disponibles:", DATOS.exists())
print("Servidor MLflow:", MLFLOW_URI)
print("Experimento:", MLFLOW_EXPERIMENTO)

Archivo de datos: /home/vickingo98/agropredict-colombia/data/processed/features_rezago_1_2.parquet
Datos disponibles: True
Servidor MLflow: http://127.0.0.1:5001
Experimento: agropredict-maiz-rezagos-t1-t2


In [2]:
# Cargar datos con ambos rezagos
df = pd.read_parquet(DATOS)

VARIABLES = [
    "semestre_b",
    "rendimiento_rezago_1",
    "rendimiento_rezago_2",
    "departamento",
    "sistema_productivo"
]

OBJETIVO = "rendimiento"

# Conservar observaciones completas para el modelado
df_modelo = df.dropna(
    subset=VARIABLES + [OBJETIVO]
).copy()

# Separación temporal
train = df_modelo[df_modelo["anio"].between(2020, 2023)].copy()
val = df_modelo[df_modelo["anio"] == 2024].copy()
test = df_modelo[df_modelo["anio"] == 2025].copy()

print("=== DISTRIBUCIÓN TEMPORAL ===")
print(f"Datos originales: {len(df):,}")
print(f"Datos utilizables: {len(df_modelo):,}")
print(f"Entrenamiento 2020–2023: {len(train):,}")
print(f"Validación 2024: {len(val):,}")
print(f"Prueba final 2025: {len(test):,}")

print("\n=== OBSERVACIONES POR AÑO ===")
print(df_modelo.groupby("anio").size())

print("\n=== VARIABLES DEL MODELO ===")
print(VARIABLES)

=== DISTRIBUCIÓN TEMPORAL ===
Datos originales: 14,935
Datos utilizables: 10,782
Entrenamiento 2020–2023: 7,334
Validación 2024: 1,713
Prueba final 2025: 1,735

=== OBSERVACIONES POR AÑO ===
anio
2020    1839
2021    1916
2022    1810
2023    1769
2024    1713
2025    1735
dtype: int64

=== VARIABLES DEL MODELO ===
['semestre_b', 'rendimiento_rezago_1', 'rendimiento_rezago_2', 'departamento', 'sistema_productivo']


In [3]:
# Ventanas expansivas para validación cruzada temporal
VENTANAS_CV = [
    ([2020], 2021),
    ([2020, 2021], 2022),
    ([2020, 2021, 2022], 2023),
]

def calcular_metricas(y_real, y_pred):
    return {
        "rmse": np.sqrt(mean_squared_error(y_real, y_pred)),
        "mae": mean_absolute_error(y_real, y_pred),
        "r2": r2_score(y_real, y_pred)
    }

resultados_baseline = []

for anios_entrenamiento, anio_validacion in VENTANAS_CV:
    datos_validacion = train[
        train["anio"] == anio_validacion
    ]

    metricas = calcular_metricas(
        datos_validacion[OBJETIVO],
        datos_validacion["rendimiento_rezago_1"]
    )

    resultados_baseline.append({
        "ventana": f"{min(anios_entrenamiento)}-{max(anios_entrenamiento)} → {anio_validacion}",
        "n_validacion": len(datos_validacion),
        **metricas
    })

df_baseline = pd.DataFrame(resultados_baseline)

print("=== BASELINE DE PERSISTENCIA POR VENTANA ===")
display(df_baseline.round(4))

print("\n=== PROMEDIO DE MÉTRICAS ===")
display(
    df_baseline[["rmse", "mae", "r2"]]
    .mean()
    .round(4)
)

=== BASELINE DE PERSISTENCIA POR VENTANA ===


,ventana,n_validacion,rmse,mae,r2
0,2020-2020 → 2021,1916,1.6935,0.7763,-0.3576
1,2020-2021 → 2022,1810,0.6633,0.3052,0.7784
2,2020-2022 → 2023,1769,0.4985,0.1978,0.8787



=== PROMEDIO DE MÉTRICAS ===


rmse    0.9518
mae     0.4264
r2      0.4332
dtype: float64

In [4]:
# Variables predictoras y objetivo
X_train = train[VARIABLES]
y_train = train[OBJETIVO]

X_val = val[VARIABLES]
y_val = val[OBJETIVO]

# Clasificación de variables
NUMERICAS = [
    "semestre_b",
    "rendimiento_rezago_1",
    "rendimiento_rezago_2"
]

CATEGORICAS = [
    "departamento",
    "sistema_productivo"
]

def crear_preprocesador():
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), NUMERICAS),
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore"),
                CATEGORICAS
            )
        ]
    )

# Comprobar dimensiones
print("=== MATRICES DE MODELADO ===")
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"y_val:   {y_val.shape}")

print("\nVariables numéricas:", NUMERICAS)
print("Variables categóricas:", CATEGORICAS)

print("\nPreprocesador preparado correctamente.")

=== MATRICES DE MODELADO ===
X_train: (7334, 5)
y_train: (7334,)
X_val:   (1713, 5)
y_val:   (1713,)

Variables numéricas: ['semestre_b', 'rendimiento_rezago_1', 'rendimiento_rezago_2']
Variables categóricas: ['departamento', 'sistema_productivo']

Preprocesador preparado correctamente.


In [5]:
def evaluar_cv_temporal(modelo, nombre_modelo):
    resultados = []

    for anios_entrenamiento, anio_validacion in VENTANAS_CV:

        # Separación temporal de cada ventana
        datos_train_cv = train[
            train["anio"].isin(anios_entrenamiento)
        ]

        datos_val_cv = train[
            train["anio"] == anio_validacion
        ]

        X_train_cv = datos_train_cv[VARIABLES]
        y_train_cv = datos_train_cv[OBJETIVO]

        X_val_cv = datos_val_cv[VARIABLES]
        y_val_cv = datos_val_cv[OBJETIVO]

        # Crear un pipeline independiente para cada ventana
        pipeline = Pipeline([
            ("preprocesador", crear_preprocesador()),
            ("modelo", modelo)
        ])

        pipeline.fit(X_train_cv, y_train_cv)

        predicciones = pipeline.predict(X_val_cv)

        metricas = calcular_metricas(
            y_val_cv,
            predicciones
        )

        resultados.append({
            "modelo": nombre_modelo,
            "anio_validacion": anio_validacion,
            "n_entrenamiento": len(datos_train_cv),
            "n_validacion": len(datos_val_cv),
            **metricas
        })

    return pd.DataFrame(resultados)


print("Función de validación temporal preparada correctamente.")

Función de validación temporal preparada correctamente.


In [6]:
# Random Forest con los hiperparámetros iniciales
rf_inicial = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

# Evaluación mediante las tres ventanas temporales
resultados_rf_inicial = evaluar_cv_temporal(
    modelo=rf_inicial,
    nombre_modelo="RandomForest_inicial"
)

print("=== RANDOM FOREST: VALIDACIÓN TEMPORAL ===")
display(resultados_rf_inicial.round(4))

print("\n=== PROMEDIO DE MÉTRICAS ===")
display(
    resultados_rf_inicial[["rmse", "mae", "r2"]]
    .mean()
    .round(4)
)

print("\n=== COMPARACIÓN DE RMSE CON PERSISTENCIA ===")

comparacion = pd.DataFrame({
    "anio": resultados_rf_inicial["anio_validacion"],
    "rmse_persistencia": df_baseline["rmse"],
    "rmse_random_forest": resultados_rf_inicial["rmse"]
})

comparacion["mejora_pct"] = (
    (comparacion["rmse_persistencia"] - comparacion["rmse_random_forest"])
    / comparacion["rmse_persistencia"]
    * 100
)

display(comparacion.round(4))

=== RANDOM FOREST: VALIDACIÓN TEMPORAL ===


,modelo,anio_validacion,n_entrenamiento,n_validacion,rmse,mae,r2
0,RandomForest_inicial,2021,1839,1916,1.4893,0.7966,-0.0499
1,RandomForest_inicial,2022,3755,1810,0.6703,0.4128,0.7737
2,RandomForest_inicial,2023,5565,1769,0.5089,0.2929,0.8736



=== PROMEDIO DE MÉTRICAS ===


rmse    0.8895
mae     0.5008
r2      0.5325
dtype: float64


=== COMPARACIÓN DE RMSE CON PERSISTENCIA ===


,anio,rmse_persistencia,rmse_random_forest,mejora_pct
0,2021,1.6935,1.4893,12.0575
1,2022,0.6633,0.6703,-1.0511
2,2023,0.4985,0.5089,-2.0757


In [7]:
# Espacio inicial de búsqueda para Random Forest
parametros_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, None],
    "min_samples_leaf": [2, 5]
}

combinaciones_rf = list(ParameterGrid(parametros_rf))

print("=== BÚSQUEDA DE HIPERPARÁMETROS RANDOM FOREST ===")
print(f"Combinaciones: {len(combinaciones_rf)}")
print(f"Ventanas temporales: {len(VENTANAS_CV)}")
print(f"Entrenamientos previstos: {len(combinaciones_rf) * len(VENTANAS_CV)}")

display(pd.DataFrame(combinaciones_rf))

=== BÚSQUEDA DE HIPERPARÁMETROS RANDOM FOREST ===
Combinaciones: 12
Ventanas temporales: 3
Entrenamientos previstos: 36


,max_depth,min_samples_leaf,n_estimators
0,10.0,2,100
1,10.0,2,200
2,10.0,2,300
3,10.0,5,100
4,10.0,5,200
5,10.0,5,300
6,NaN,2,100
7,NaN,2,200
8,NaN,2,300
9,NaN,5,100


In [8]:
from time import perf_counter

resultados_busqueda_rf = []
inicio = perf_counter()

for i, params in enumerate(combinaciones_rf, start=1):

    modelo = RandomForestRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )

    resultados_cv = evaluar_cv_temporal(
        modelo=modelo,
        nombre_modelo="RandomForest"
    )

    resultados_busqueda_rf.append({
        **params,
        "rmse_cv": resultados_cv["rmse"].mean(),
        "mae_cv": resultados_cv["mae"].mean(),
        "r2_cv": resultados_cv["r2"].mean(),
        "rmse_2021": resultados_cv.loc[
            resultados_cv["anio_validacion"] == 2021, "rmse"
        ].iloc[0],
        "rmse_2022": resultados_cv.loc[
            resultados_cv["anio_validacion"] == 2022, "rmse"
        ].iloc[0],
        "rmse_2023": resultados_cv.loc[
            resultados_cv["anio_validacion"] == 2023, "rmse"
        ].iloc[0]
    })

    print(
        f"[{i:02d}/{len(combinaciones_rf)}] "
        f"RMSE promedio: {resultados_cv['rmse'].mean():.4f}"
    )

df_busqueda_rf = (
    pd.DataFrame(resultados_busqueda_rf)
    .sort_values("rmse_cv")
    .reset_index(drop=True)
)

mejores_parametros_rf = {
    clave: combinaciones_rf[
        next(
            i for i, p in enumerate(combinaciones_rf)
            if all(
                p[k] == df_busqueda_rf.iloc[0][k]
                or (p[k] is None and pd.isna(df_busqueda_rf.iloc[0][k]))
                for k in params
            )
        )
    ][clave]
    for clave in parametros_rf
}

print("\n=== MEJORES COMBINACIONES RANDOM FOREST ===")
display(df_busqueda_rf.head(12).round(4))

print("\n=== MEJORES HIPERPARÁMETROS ===")
print(mejores_parametros_rf)

print(f"\nTiempo total: {perf_counter() - inicio:.1f} segundos")

[01/12] RMSE promedio: 0.9101
[02/12] RMSE promedio: 0.9077
[03/12] RMSE promedio: 0.9072
[04/12] RMSE promedio: 0.8825
[05/12] RMSE promedio: 0.8825
[06/12] RMSE promedio: 0.8813
[07/12] RMSE promedio: 0.9296
[08/12] RMSE promedio: 0.9267
[09/12] RMSE promedio: 0.9262
[10/12] RMSE promedio: 0.8893
[11/12] RMSE promedio: 0.8895
[12/12] RMSE promedio: 0.8885

=== MEJORES COMBINACIONES RANDOM FOREST ===


,max_depth,min_samples_leaf,n_estimators,rmse_cv,mae_cv,r2_cv,rmse_2021,rmse_2022,rmse_2023
0,10.0,5,300,0.8813,0.4915,0.5381,1.4866,0.6573,0.5000
1,10.0,5,100,0.8825,0.4927,0.5374,1.4863,0.6607,0.5004
2,10.0,5,200,0.8825,0.4925,0.5368,1.4885,0.6592,0.4997
3,NaN,5,300,0.8885,0.5001,0.5336,1.4874,0.6689,0.5094
4,NaN,5,100,0.8893,0.5009,0.5331,1.4872,0.6712,0.5096
5,NaN,5,200,0.8895,0.5008,0.5325,1.4893,0.6703,0.5089
6,10.0,2,300,0.9072,0.5005,0.5109,1.5299,0.6718,0.5198
7,10.0,2,200,0.9077,0.5011,0.5109,1.5286,0.6745,0.5201
8,10.0,2,100,0.9101,0.5019,0.5082,1.5330,0.6763,0.5210
9,NaN,2,300,0.9262,0.5166,0.4986,1.5322,0.7021,0.5444



=== MEJORES HIPERPARÁMETROS ===
{'n_estimators': 300, 'max_depth': 10, 'min_samples_leaf': 5}

Tiempo total: 23.7 segundos


In [10]:
# Registrar correctamente los resultados ya calculados
with mlflow.start_run(run_name="RandomForest_CV_temporal_t1_t2"):

    mlflow.log_params({
        "modelo": "RandomForest",
        **mejores_parametros_rf,
        "random_state": 42,
        "tipo_validacion": "expanding_window",
        "numero_folds": 3,
        "variables_rezago": "t-1_t-2"
    })

    mlflow.log_metrics({
        "cv_rmse": float(df_busqueda_rf.iloc[0]["rmse_cv"]),
        "cv_mae": float(df_busqueda_rf.iloc[0]["mae_cv"]),
        "cv_r2": float(df_busqueda_rf.iloc[0]["r2_cv"]),
        "rmse_2021": float(df_busqueda_rf.iloc[0]["rmse_2021"]),
        "rmse_2022": float(df_busqueda_rf.iloc[0]["rmse_2022"]),
        "rmse_2023": float(df_busqueda_rf.iloc[0]["rmse_2023"])
    })

    # Registrar directamente el DataFrame
    mlflow.log_table(
        df_busqueda_rf,
        artifact_file="busqueda_hiperparametros_rf.json"
    )

    print("Random Forest registrado correctamente en MLflow.")

Random Forest registrado correctamente en MLflow.
🏃 View run RandomForest_CV_temporal_t1_t2 at: http://127.0.0.1:5001/#/experiments/2/runs/56d9daa9e0a549a697982316d7534a88
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


In [11]:
# Espacio de búsqueda para XGBoost
parametros_xgb = {
    "n_estimators": [200, 400],
    "max_depth": [3, 5],
    "learning_rate": [0.03, 0.07],
    "subsample": [0.8, 1.0]
}

combinaciones_xgb = list(ParameterGrid(parametros_xgb))

print("=== BÚSQUEDA DE HIPERPARÁMETROS XGBOOST ===")
print(f"Combinaciones: {len(combinaciones_xgb)}")
print(f"Ventanas temporales: {len(VENTANAS_CV)}")
print(f"Entrenamientos previstos: {len(combinaciones_xgb) * len(VENTANAS_CV)}")

display(pd.DataFrame(combinaciones_xgb))

=== BÚSQUEDA DE HIPERPARÁMETROS XGBOOST ===
Combinaciones: 16
Ventanas temporales: 3
Entrenamientos previstos: 48


,learning_rate,max_depth,n_estimators,subsample
0,0.03,3,200,0.8
1,0.03,3,200,1.0
2,0.03,3,400,0.8
3,0.03,3,400,1.0
4,0.03,5,200,0.8
5,0.03,5,200,1.0
6,0.03,5,400,0.8
7,0.03,5,400,1.0
8,0.07,3,200,0.8
9,0.07,3,200,1.0


In [12]:
from time import perf_counter

resultados_busqueda_xgb = []
inicio = perf_counter()

for i, params in enumerate(combinaciones_xgb, start=1):

    modelo = XGBRegressor(
        **params,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
        verbosity=0
    )

    resultados_cv = evaluar_cv_temporal(
        modelo=modelo,
        nombre_modelo="XGBoost"
    )

    resultados_busqueda_xgb.append({
        **params,
        "rmse_cv": resultados_cv["rmse"].mean(),
        "mae_cv": resultados_cv["mae"].mean(),
        "r2_cv": resultados_cv["r2"].mean(),
        "rmse_2021": resultados_cv.loc[
            resultados_cv["anio_validacion"] == 2021, "rmse"
        ].iloc[0],
        "rmse_2022": resultados_cv.loc[
            resultados_cv["anio_validacion"] == 2022, "rmse"
        ].iloc[0],
        "rmse_2023": resultados_cv.loc[
            resultados_cv["anio_validacion"] == 2023, "rmse"
        ].iloc[0]
    })

    print(
        f"[{i:02d}/{len(combinaciones_xgb)}] "
        f"RMSE promedio: {resultados_cv['rmse'].mean():.4f}"
    )

df_busqueda_xgb = (
    pd.DataFrame(resultados_busqueda_xgb)
    .sort_values("rmse_cv")
    .reset_index(drop=True)
)

# Recuperar los mejores parámetros conservando sus tipos originales
mejor_fila = df_busqueda_xgb.iloc[0]

mejores_parametros_xgb = {
    "learning_rate": float(mejor_fila["learning_rate"]),
    "max_depth": int(mejor_fila["max_depth"]),
    "n_estimators": int(mejor_fila["n_estimators"]),
    "subsample": float(mejor_fila["subsample"])
}

print("\n=== RESULTADOS DE OPTIMIZACIÓN XGBOOST ===")
display(df_busqueda_xgb.round(4))

print("\n=== MEJORES HIPERPARÁMETROS ===")
print(mejores_parametros_xgb)

print(f"\nTiempo total: {perf_counter() - inicio:.1f} segundos")

[01/16] RMSE promedio: 0.9056
[02/16] RMSE promedio: 0.9049
[03/16] RMSE promedio: 0.9191
[04/16] RMSE promedio: 0.9131
[05/16] RMSE promedio: 0.9217
[06/16] RMSE promedio: 0.9216
[07/16] RMSE promedio: 0.9436
[08/16] RMSE promedio: 0.9415
[09/16] RMSE promedio: 0.9205
[10/16] RMSE promedio: 0.9194
[11/16] RMSE promedio: 0.9378
[12/16] RMSE promedio: 0.9411
[13/16] RMSE promedio: 0.9457
[14/16] RMSE promedio: 0.9474
[15/16] RMSE promedio: 0.9714
[16/16] RMSE promedio: 0.9764

=== RESULTADOS DE OPTIMIZACIÓN XGBOOST ===


,learning_rate,max_depth,n_estimators,subsample,rmse_cv,mae_cv,r2_cv,rmse_2021,rmse_2022,rmse_2023
0,0.03,3,200,1.0,0.9049,0.5177,0.4949,1.5913,0.6153,0.5080
1,0.03,3,200,0.8,0.9056,0.5149,0.4931,1.5955,0.6154,0.5060
2,0.03,3,400,1.0,0.9131,0.5175,0.4812,1.6191,0.6173,0.5030
3,0.03,3,400,0.8,0.9191,0.5159,0.4698,1.6438,0.6141,0.4993
4,0.07,3,200,1.0,0.9194,0.5192,0.4726,1.6346,0.6213,0.5021
5,0.07,3,200,0.8,0.9205,0.5172,0.4687,1.6449,0.6138,0.5028
6,0.03,5,200,1.0,0.9216,0.5111,0.4690,1.6405,0.6297,0.4944
7,0.03,5,200,0.8,0.9217,0.5117,0.4677,1.6443,0.6260,0.4949
8,0.07,3,400,0.8,0.9378,0.5250,0.4451,1.6852,0.6234,0.5048
9,0.07,3,400,1.0,0.9411,0.5277,0.4479,1.6704,0.6449,0.5081



=== MEJORES HIPERPARÁMETROS ===
{'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 200, 'subsample': 1.0}

Tiempo total: 8.4 segundos


In [13]:
with mlflow.start_run(run_name="XGBoost_CV_temporal_t1_t2"):

    mlflow.log_params({
        "modelo": "XGBoost",
        **mejores_parametros_xgb,
        "colsample_bytree": 0.8,
        "objective": "reg:squarederror",
        "random_state": 42,
        "tipo_validacion": "expanding_window",
        "numero_folds": 3,
        "variables_rezago": "t-1_t-2"
    })

    mlflow.log_metrics({
        "cv_rmse": float(df_busqueda_xgb.iloc[0]["rmse_cv"]),
        "cv_mae": float(df_busqueda_xgb.iloc[0]["mae_cv"]),
        "cv_r2": float(df_busqueda_xgb.iloc[0]["r2_cv"]),
        "rmse_2021": float(df_busqueda_xgb.iloc[0]["rmse_2021"]),
        "rmse_2022": float(df_busqueda_xgb.iloc[0]["rmse_2022"]),
        "rmse_2023": float(df_busqueda_xgb.iloc[0]["rmse_2023"])
    })

    # Guardar la tabla completa de combinaciones
    mlflow.log_table(
        df_busqueda_xgb,
        artifact_file="busqueda_hiperparametros_xgb.json"
    )

    print("XGBoost registrado correctamente en MLflow.")

XGBoost registrado correctamente en MLflow.
🏃 View run XGBoost_CV_temporal_t1_t2 at: http://127.0.0.1:5001/#/experiments/2/runs/ca8e0afaef174ef08acf121d29b4a064
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


In [14]:
# Construir modelos con los mejores hiperparámetros
modelos_optimizados = {
    "RandomForest_optimizado": RandomForestRegressor(
        **mejores_parametros_rf,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost_optimizado": XGBRegressor(
        **mejores_parametros_xgb,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
        verbosity=0
    )
}

resultados_validacion_2024 = []
pipelines_optimizados = {}

for nombre, modelo in modelos_optimizados.items():

    pipeline = Pipeline([
        ("preprocesador", crear_preprocesador()),
        ("modelo", modelo)
    ])

    # Entrenar exclusivamente con 2020-2023
    pipeline.fit(X_train, y_train)

    # Evaluar sobre 2024
    predicciones = pipeline.predict(X_val)
    metricas = calcular_metricas(y_val, predicciones)

    resultados_validacion_2024.append({
        "modelo": nombre,
        "n_validacion": len(y_val),
        **metricas
    })

    pipelines_optimizados[nombre] = pipeline

# Baseline de persistencia sobre las mismas observaciones
metricas_baseline_2024 = calcular_metricas(
    y_val,
    val["rendimiento_rezago_1"]
)

resultados_validacion_2024.append({
    "modelo": "Persistencia_t1",
    "n_validacion": len(y_val),
    **metricas_baseline_2024
})

df_validacion_2024 = (
    pd.DataFrame(resultados_validacion_2024)
    .sort_values("rmse")
    .reset_index(drop=True)
)

print("=== COMPARACIÓN DE MODELOS — VALIDACIÓN 2024 ===")
display(df_validacion_2024.round(4))

print("\nMejor modelo según RMSE:")
print(df_validacion_2024.iloc[0]["modelo"])

=== COMPARACIÓN DE MODELOS — VALIDACIÓN 2024 ===


,modelo,n_validacion,rmse,mae,r2
0,XGBoost_optimizado,1713,0.5213,0.2914,0.8607
1,RandomForest_optimizado,1713,0.5233,0.2723,0.8597
2,Persistencia_t1,1713,0.5672,0.2209,0.8351



Mejor modelo según RMSE:
XGBoost_optimizado


In [15]:
with mlflow.start_run(run_name="Comparacion_modelos_validacion_2024"):

    mlflow.log_params({
        "periodo_entrenamiento": "2020-2023",
        "periodo_validacion": "2024",
        "criterio_seleccion": "rmse",
        "modelo_seleccionado": "XGBoost_optimizado",
        "tipo_rezagos": "t-1_t-2"
    })

    for _, fila in df_validacion_2024.iterrows():

        nombre = fila["modelo"]

        mlflow.log_metrics({
            f"{nombre}_rmse": float(fila["rmse"]),
            f"{nombre}_mae": float(fila["mae"]),
            f"{nombre}_r2": float(fila["r2"])
        })

    mlflow.log_table(
        df_validacion_2024,
        artifact_file="comparacion_validacion_2024.json"
    )

    print("Comparación de modelos registrada correctamente en MLflow.")

Comparación de modelos registrada correctamente en MLflow.
🏃 View run Comparacion_modelos_validacion_2024 at: http://127.0.0.1:5001/#/experiments/2/runs/84f903f8ac284f36b6f3f93a018e0bc3
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


In [16]:
# Entrenamiento final con datos de 2020 a 2024
datos_entrenamiento_final = df_modelo[
    df_modelo["anio"].between(2020, 2024)
].copy()

X_entrenamiento_final = datos_entrenamiento_final[VARIABLES]
y_entrenamiento_final = datos_entrenamiento_final[OBJETIVO]

X_test = test[VARIABLES]
y_test = test[OBJETIVO]

# Modelo seleccionado con hiperparámetros fijos
modelo_xgb_final = Pipeline([
    ("preprocesador", crear_preprocesador()),
    ("modelo", XGBRegressor(
        **mejores_parametros_xgb,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
        verbosity=0
    ))
])

# Entrenar y predecir
modelo_xgb_final.fit(
    X_entrenamiento_final,
    y_entrenamiento_final
)

predicciones_2025 = modelo_xgb_final.predict(X_test)

# Evaluación final
metricas_xgb_2025 = calcular_metricas(
    y_test,
    predicciones_2025
)

# Baseline sobre las mismas observaciones
metricas_persistencia_2025 = calcular_metricas(
    y_test,
    test["rendimiento_rezago_1"]
)

df_evaluacion_2025 = pd.DataFrame([
    {
        "modelo": "XGBoost_optimizado",
        **metricas_xgb_2025
    },
    {
        "modelo": "Persistencia_t1",
        **metricas_persistencia_2025
    }
]).sort_values("rmse").reset_index(drop=True)

print("=== EVALUACIÓN FINAL — 2025 ===")
print(f"Observaciones de entrenamiento: {len(X_entrenamiento_final):,}")
print(f"Observaciones de prueba: {len(X_test):,}")

display(df_evaluacion_2025.round(4))

mejora_rmse = (
    (metricas_persistencia_2025["rmse"] - metricas_xgb_2025["rmse"])
    / metricas_persistencia_2025["rmse"]
    * 100
)

print(f"\nMejora de RMSE frente a persistencia: {mejora_rmse:.2f}%")

=== EVALUACIÓN FINAL — 2025 ===
Observaciones de entrenamiento: 9,047
Observaciones de prueba: 1,735


,modelo,rmse,mae,r2
0,XGBoost_optimizado,0.3708,0.2160,0.9275
1,Persistencia_t1,0.4140,0.1478,0.9097



Mejora de RMSE frente a persistencia: 10.42%


In [17]:
with mlflow.start_run(run_name="XGBoost_evaluacion_final_2025_t1_t2"):

    mlflow.log_params({
        "modelo": "XGBoost",
        **mejores_parametros_xgb,
        "colsample_bytree": 0.8,
        "random_state": 42,
        "periodo_entrenamiento": "2020-2024",
        "periodo_prueba": "2025",
        "tipo_rezagos": "t-1_t-2",
        "criterio_seleccion": "rmse"
    })

    mlflow.log_metrics({
        "test_2025_rmse": float(metricas_xgb_2025["rmse"]),
        "test_2025_mae": float(metricas_xgb_2025["mae"]),
        "test_2025_r2": float(metricas_xgb_2025["r2"]),
        "baseline_2025_rmse": float(metricas_persistencia_2025["rmse"]),
        "baseline_2025_mae": float(metricas_persistencia_2025["mae"]),
        "baseline_2025_r2": float(metricas_persistencia_2025["r2"]),
        "mejora_rmse_pct": float(mejora_rmse)
    })

    mlflow.log_table(
        df_evaluacion_2025,
        artifact_file="evaluacion_final_2025.json"
    )

    print("Evaluación final de XGBoost registrada correctamente en MLflow.")

Evaluación final de XGBoost registrada correctamente en MLflow.
🏃 View run XGBoost_evaluacion_final_2025_t1_t2 at: http://127.0.0.1:5001/#/experiments/2/runs/caf8a92ca16a4aa5b826889823a437c6
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


In [18]:
# Preparación del conjunto de entrenamiento para producción
datos_produccion = df_modelo[
    df_modelo["anio"].between(2020, 2025)
].copy()

X_produccion = datos_produccion[VARIABLES]
y_produccion = datos_produccion[OBJETIVO]

print("=== DATOS PARA MODELO DE PRODUCCIÓN ===")
print(f"Observaciones: {len(datos_produccion):,}")
print(f"Período: {datos_produccion['anio'].min()}–{datos_produccion['anio'].max()}")

print("\n=== VALORES FALTANTES ===")
print(X_produccion.isna().sum())

print("\n=== DISTRIBUCIÓN POR AÑO ===")
print(datos_produccion.groupby("anio").size())

print("\n=== HIPERPARÁMETROS SELECCIONADOS ===")
print(mejores_parametros_xgb)

=== DATOS PARA MODELO DE PRODUCCIÓN ===
Observaciones: 10,782
Período: 2020–2025

=== VALORES FALTANTES ===
semestre_b              0
rendimiento_rezago_1    0
rendimiento_rezago_2    0
departamento            0
sistema_productivo      0
dtype: int64

=== DISTRIBUCIÓN POR AÑO ===
anio
2020    1839
2021    1916
2022    1810
2023    1769
2024    1713
2025    1735
dtype: int64

=== HIPERPARÁMETROS SELECCIONADOS ===
{'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 200, 'subsample': 1.0}


In [19]:
from time import perf_counter

# Pipeline definitivo de producción
modelo_xgb_produccion = Pipeline([
    ("preprocesador", crear_preprocesador()),
    ("modelo", XGBRegressor(
        **mejores_parametros_xgb,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
        verbosity=0
    ))
])

# Entrenamiento con todos los datos disponibles
inicio = perf_counter()

modelo_xgb_produccion.fit(
    X_produccion,
    y_produccion
)

tiempo_entrenamiento = perf_counter() - inicio

print("=== MODELO DE PRODUCCIÓN ===")
print("Entrenamiento completado correctamente.")
print(f"Observaciones utilizadas: {len(X_produccion):,}")
print(f"Variables predictoras: {len(VARIABLES)}")
print(f"Tiempo de entrenamiento: {tiempo_entrenamiento:.2f} segundos")

print("\n=== CONFIGURACIÓN DEL MODELO ===")
print(modelo_xgb_produccion.named_steps["modelo"].get_params())

=== MODELO DE PRODUCCIÓN ===
Entrenamiento completado correctamente.
Observaciones utilizadas: 10,782
Variables predictoras: 5
Tiempo de entrenamiento: 0.39 segundos

=== CONFIGURACIÓN DEL MODELO ===
{'objective': 'reg:squarederror', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 0.8, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': True, 'eval_metric': None, 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.03, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 3, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 200, 'n_jobs': -1, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None

In [20]:
import skops.io as sio
from tempfile import TemporaryDirectory

with mlflow.start_run(run_name="XGBoost_modelo_produccion_t1_t2") as run:

    mlflow.log_params({
        "modelo": "XGBoost",
        **mejores_parametros_xgb,
        "colsample_bytree": 0.8,
        "random_state": 42,
        "periodo_entrenamiento": "2020-2025",
        "observaciones_entrenamiento": len(X_produccion),
        "tipo_rezagos": "t-1_t-2",
        "version": "produccion_t1_t2"
    })

    mlflow.log_metric(
        "tiempo_entrenamiento_segundos",
        tiempo_entrenamiento
    )

    # Guardar el pipeline completo en formato skops
    with TemporaryDirectory() as directorio:
        ruta_modelo = Path(directorio) / "xgboost_maiz_t1_t2.skops"

        sio.dump(modelo_xgb_produccion, ruta_modelo)

        mlflow.log_artifact(
            str(ruta_modelo),
            artifact_path="modelo"
        )

    print("=== MODELO REGISTRADO EN MLFLOW ===")
    print(f"Run ID: {run.info.run_id}")
    print("Artefacto: modelo/xgboost_maiz_t1_t2.skops")
    print("Experimento:", MLFLOW_EXPERIMENTO)

=== MODELO REGISTRADO EN MLFLOW ===
Run ID: 61419ed6d92046b9ae6a3bd64001e463
Artefacto: modelo/xgboost_maiz_t1_t2.skops
Experimento: agropredict-maiz-rezagos-t1-t2
🏃 View run XGBoost_modelo_produccion_t1_t2 at: http://127.0.0.1:5001/#/experiments/2/runs/61419ed6d92046b9ae6a3bd64001e463
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


In [21]:
import skops.io as sio
from mlflow.artifacts import download_artifacts

# Identificador de la ejecución del modelo de producción
RUN_ID_PRODUCCION = "61419ed6d92046b9ae6a3bd64001e463"

# Descargar el artefacto desde MLflow
ruta_recuperada = download_artifacts(
    run_id=RUN_ID_PRODUCCION,
    artifact_path="modelo/xgboost_maiz_t1_t2.skops"
)

print("=== RECUPERACIÓN DEL MODELO ===")
print("Archivo:", ruta_recuperada)
print("Existe:", Path(ruta_recuperada).is_file())

# Inspeccionar los tipos antes de cargar el modelo
tipos_no_confiables = sio.get_untrusted_types(
    file=ruta_recuperada
)

print("\n=== TIPOS QUE REQUIEREN REVISIÓN ===")
print(tipos_no_confiables)

if tipos_no_confiables:
    print("\nRevisar estos tipos antes de cargar el modelo.")
else:
    # Cargar el pipeline recuperado
    modelo_recuperado = sio.load(ruta_recuperada)

    # Comparar predicciones con el modelo original
    muestra = X_produccion.head(10)

    pred_original = modelo_xgb_produccion.predict(muestra)
    pred_recuperada = modelo_recuperado.predict(muestra)

    print("\n=== VERIFICACIÓN DE PREDICCIONES ===")
    print("Predicciones coincidentes:",
          np.allclose(pred_original, pred_recuperada))

    print("Diferencia máxima:",
          np.max(np.abs(pred_original - pred_recuperada)))

    display(pd.DataFrame({
        "prediccion_original": pred_original,
        "prediccion_recuperada": pred_recuperada
    }).round(6))

=== RECUPERACIÓN DEL MODELO ===
Archivo: /tmp/tmpoz2gkkk9/xgboost_maiz_t1_t2.skops
Existe: True

=== TIPOS QUE REQUIEREN REVISIÓN ===
['xgboost.core.Booster', 'xgboost.sklearn.XGBRegressor']

Revisar estos tipos antes de cargar el modelo.


In [22]:
# Autorizar únicamente los tipos de XGBoost identificados
tipos_autorizados = [
    "xgboost.core.Booster",
    "xgboost.sklearn.XGBRegressor"
]

# Verificar que no existan otros tipos inesperados
assert set(tipos_no_confiables) == set(tipos_autorizados), (
    f"Se encontraron tipos adicionales: {tipos_no_confiables}"
)

# Cargar el modelo recuperado desde MLflow
modelo_recuperado = sio.load(
    ruta_recuperada,
    trusted=tipos_autorizados
)

# Comparar las predicciones de ambos modelos
muestra = X_produccion.head(10)

pred_original = modelo_xgb_produccion.predict(muestra)
pred_recuperada = modelo_recuperado.predict(muestra)

coinciden = np.allclose(
    pred_original,
    pred_recuperada,
    rtol=1e-6,
    atol=1e-8
)

diferencia_maxima = np.max(
    np.abs(pred_original - pred_recuperada)
)

print("=== VERIFICACIÓN DEL MODELO RECUPERADO ===")
print("Modelo cargado correctamente.")
print(f"Predicciones coincidentes: {coinciden}")
print(f"Diferencia máxima: {diferencia_maxima:.10f}")

display(pd.DataFrame({
    "prediccion_original": pred_original,
    "prediccion_recuperada": pred_recuperada
}).round(6))

assert coinciden, "Las predicciones no coinciden."

=== VERIFICACIÓN DEL MODELO RECUPERADO ===
Modelo cargado correctamente.
Predicciones coincidentes: True
Diferencia máxima: 0.0000000000


,prediccion_original,prediccion_recuperada
0,1.962091,1.962091
1,1.956398,1.956398
2,1.962091,1.962091
3,1.956398,1.956398
4,1.962091,1.962091
5,1.956398,1.956398
6,1.962091,1.962091
7,1.956398,1.956398
8,1.962091,1.962091
9,1.956398,1.956398


In [23]:
import shutil

ruta_destino = PROYECTO / "models" / "xgboost_maiz_t1_t2.skops"

# Evitar sobrescribir accidentalmente un modelo existente
if ruta_destino.exists():
    raise FileExistsError(
        f"El archivo ya existe: {ruta_destino}"
    )

# Copiar el artefacto verificado desde MLflow
shutil.copy2(ruta_recuperada, ruta_destino)

print("=== MODELO PREPARADO PARA VERSIONAMIENTO ===")
print("Archivo:", ruta_destino)
print("Existe:", ruta_destino.is_file())
print("Tamaño:", round(ruta_destino.stat().st_size / 1024, 2), "KB")

=== MODELO PREPARADO PARA VERSIONAMIENTO ===
Archivo: /home/vickingo98/agropredict-colombia/models/xgboost_maiz_t1_t2.skops
Existe: True
Tamaño: 352.44 KB
